# 07.1 - Knowledge Graph - GraphRAG

This notebook uses Microsoft's [GraphRAG](https://microsoft.github.io/graphrag/) to build a knowledge graph over a small corpus of paper abstracts and then run a graph-aware retrieval query against it. GraphRAG is invoked here through its CLI rather than its Python API.

In [ ]:
%pip install graphrag

## Notebook Setup

Install the GraphRAG CLI and Python package.

In [ ]:
from genscai import paths

project_dir = paths.output / "graphrag"
data_dir = project_dir / "input"

Define the project and input directories that GraphRAG will use. Each abstract becomes a separate `.txt` file under `input/`.

In [ ]:
import os

os.mkdir(project_dir)
os.mkdir(data_dir)

Create the project and input directories on disk.

In [ ]:
!graphrag init --root $project_dir

Bootstrap a default GraphRAG project skeleton (config, prompts, output directory) under `project_dir`.

In [ ]:
%%writefile $project_dir/.env
GRAPHRAG_API_KEY=...

Drop a `.env` file into the project directory containing the API key for the LLM that GraphRAG will call. Replace `...` with your real key before running indexing.

In [ ]:
import json

with open(paths.data / "training_modeling_papers.json") as fin:
    papers = json.load(fin)

for i, paper in enumerate(papers):
    with open(data_dir / f"{i}.txt", "w") as f:
        f.write(paper["abstract"])

Write each training-paper abstract into its own `.txt` file under the input directory so GraphRAG can ingest them as documents.

In [ ]:
!graphrag index --root $project_dir

Run GraphRAG's indexer. This step issues many LLM calls to extract entities, relationships, and community summaries; expect it to take a while and incur API costs proportional to the corpus size.

In [ ]:
!graphrag query \
--root $project_dir \
--method local \
--query "List all known disease modeling techniques"

Issue a local-search query against the indexed graph. `--method local` answers from the entities and relationships discovered during indexing rather than running global community-level summarization.